# Discourse-Grounded Linguistic Marker Analysis — GSM8K CoT Reasoning

**Tujuan:** Menganalisis gaya reasoning LLM menggunakan marker linguistik berbasis wacana (discourse-grounded),  
bukan sekadar penghitungan kata. Fokus pada **`response_content`** sebagai discourse artifact.

**Taksonomi:** 8 grup, 32 marker (25 marker awal + 7 tambahan dari audit crosscheck sumber)  
**Analisis:** Per model, per persona, gender (man/woman/neutral), implicit vs explicit

---
**Dataset:** GSM8K | 14 Model | 9 Persona Conditions | ~149K respons  
**Referensi:** Hyland (2005) metadiscourse; Biber (1988) register; Wei et al. (2022) CoT; Park & Lei (2026)


In [ ]:
# ── Install (run once) ──────────────────────────────────────────────────────
# !pip install pandas numpy matplotlib seaborn scipy --quiet

import re
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from pathlib import Path

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", font_scale=0.85)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.4f}".format)

print("✅ Libraries loaded")


In [ ]:
# ── Load CSV ────────────────────────────────────────────────────────────────
DATA_PATH = Path("all_results_gsm8k_lite.csv")   # ← sesuaikan path jika perlu

df = pd.read_csv(DATA_PATH, low_memory=False)
df = df[df["persona"] != "_"].copy()
df["text"] = df["response_content"].fillna("").astype(str)
df["is_correct"] = df["is_correct"].map({True:True, False:False, "True":True, "False":False})

# ── Model metadata ───────────────────────────────────────────────────────────
META = {
    "allenai_olmo-3-32b-think":              ("OLMo-3 Think 32B",  "32B",         "Dense","Thinking",          "Medium (15–50B)",      "AllenAI"),
    "allenai_olmo-3.1-32b-instruct":         ("OLMo-3.1 32B",      "32B",         "Dense","Instruct",          "Medium (15–50B)",      "AllenAI"),
    "deepseek_deepseek-r1-distill-qwen-32b": ("DeepSeek R1 32B",   "32B",         "Dense","Thinking",          "Medium (15–50B)",      "DeepSeek"),
    "deepseek_deepseek-v3.2":                ("DeepSeek V3.2",      "~236B MoE",   "MoE",  "Instruct/Reasoning","Ultra Large (200B+)",  "DeepSeek"),
    "google_gemini-2.5-flash":               ("Gemini 2.5 Flash",  "Proprietary", "Dense","Fast/General",      "Proprietary",          "Google"),
    "google_gemma-2-9b-it":                  ("Gemma-2 9B",         "9B",          "Dense","Instruct",          "Small (7–10B)",        "Google"),
    "google_gemma-3-27b-it":                 ("Gemma-3 27B",        "27B",         "Dense","Instruct",          "Medium (15–50B)",      "Google"),
    "meta-llama_llama-3.1-8b-instruct":      ("LLaMA 3.1 8B",      "8B",          "Dense","Instruct",          "Small (7–10B)",        "Meta"),
    "meta-llama_llama-3.3-70b-instruct":     ("LLaMA 3.3 70B",     "70B",         "Dense","Instruct",          "Large (70–100B)",      "Meta"),
    "mistralai_ministral-8b-2512":           ("Ministral 8B",       "8B",          "Dense","Instruct",          "Small (7–10B)",        "Mistral"),
    "openai_gpt-oss-120b":                   ("GPT-oss 120B",       "120B",        "Dense","Instruct",          "Very Large (100–200B)","OpenAI"),
    "openai_gpt-oss-20b":                    ("GPT-oss 20B",        "20B",         "Dense","Instruct",          "Medium (15–50B)",      "OpenAI"),
    "qwen_qwen3-235b-a22b-2507":             ("Qwen3 235B",         "235B MoE A22B","MoE", "Instruct",          "Ultra Large (200B+)",  "Alibaba"),
    "qwen_qwen3-next-80b-a3b-thinking":      ("Qwen3-Think 80B",   "80B MoE A3B", "MoE",  "Thinking",          "Large (70–100B)",      "Alibaba"),
}

for k,(lbl,params,arch,mtype,scat,prov) in META.items():
    mask = df["model"] == k
    df.loc[mask,"model_label"] = lbl
    df.loc[mask,"arch"]        = arch
    df.loc[mask,"model_type"]  = mtype
    df.loc[mask,"size_cat"]    = scat
    df.loc[mask,"provider"]    = prov

# ── Persona groupings ─────────────────────────────────────────────────────────
PERSONA_GROUPS = {
    "neutral":         ["neutral"],
    "man_explicit":    ["man_explicit_1"],
    "man_implicit":    ["man_implicit_1","man_implicit_2","man_implicit_3"],
    "woman_explicit":  ["woman_explicit_1"],
    "woman_implicit":  ["woman_implicit_1","woman_implicit_2","woman_implicit_3"],
    "man_all":         ["man_explicit_1","man_implicit_1","man_implicit_2","man_implicit_3"],
    "woman_all":       ["woman_explicit_1","woman_implicit_1","woman_implicit_2","woman_implicit_3"],
    "explicit_all":    ["man_explicit_1","woman_explicit_1"],
    "implicit_all":    ["man_implicit_1","man_implicit_2","man_implicit_3",
                        "woman_implicit_1","woman_implicit_2","woman_implicit_3"],
}

def persona_group(p):
    for g, ps in PERSONA_GROUPS.items():
        if p in ps and g not in ("man_all","woman_all","explicit_all","implicit_all"):
            return g
    return "other"

df["persona_group"] = df["persona"].apply(persona_group)

print(f"✅ Data loaded: {len(df):,} rows | {df['model_label'].nunique()} models")
print(f"   Personas: {sorted(df['persona'].unique())}")
print(f"   Persona groups: {sorted(df['persona_group'].unique())}")


## 1. Discourse Zoning

Setiap respons dipecah menjadi tiga zona sebelum ekstraksi marker.
Marker yang discourse function-nya bergantung pada **lokasi** diekstrak dari zona yang tepat:
- `opening_zone` → opener rituals (A1, A3)
- `reasoning_zone` → epistemic stance, mid-reasoning revision (C1, F2)
- `closure_zone` → verifikasi closure, formal/NL closure (F1, F4, F5)
- `full_text` → marker yang bersifat global (B-D markers, E markers)


In [ ]:
# ── Text Zoning ──────────────────────────────────────────────────────────────

def split_zones(text: str):
    """
    Pisahkan teks menjadi tiga zona discourse:
      opening_zone   : 2 kalimat pertama atau ~300 char — tempat opening ritual
      reasoning_zone : tengah teks — tempat epistemic stance & revision
      closure_zone   : 2 kalimat terakhir atau ~300 char — tempat verifikasi/closure
    """
    if not isinstance(text, str) or not text.strip():
        return "", "", ""
    text = text.strip()
    # Pisahkan per kalimat atau per paragraf
    sents = re.split(r"(?<=[.!?])\s+|\n\n+", text)
    sents = [s for s in sents if s.strip()]

    if len(sents) <= 2:
        return text, "", text

    opening  = " ".join(sents[:2])[:300]
    closure  = " ".join(sents[-2:])[-300:]
    full_len = len(text)
    # reasoning zone: setelah opening_zone, sebelum closure_zone
    reasoning = text[min(300, full_len // 3) : max(0, full_len - 200)]
    return opening, reasoning, closure

print("Splitting zones...")
df[["opening_zone","reasoning_zone","closure_zone"]] = df["text"].apply(
    lambda t: pd.Series(split_zones(t))
)

# Word count dari full text
df["word_count"] = df["text"].apply(
    lambda t: max(len(re.findall(r"\b\w+\b", t)), 1)
)

print(f"✅ Zones created. Sample word counts:")
print(df.groupby("model_label")["word_count"].mean().round(1).sort_values(ascending=False))


## 2. Taksonomi Lengkap — Discourse-Grounded Marker (32 marker, 8 grup)

Taksonomi direvisi berdasarkan crosscheck dengan:
- **Hyland (2005)** — Metadiscourse: Exploring Interaction in Writing
- **Biber (1988, 1995)** — Multi-Dimensional Analysis of English
- **Wei et al. (2022)** — Chain-of-Thought Prompting (NeurIPS)
- **Park & Lei (2026)** — Logical Connectives as High-Entropy Forking Points
- **Meek et al. (2025)** — Measuring CoT Monitorability Through Verbosity
- **arXiv:2602.09832** — LLM Reasoning Predicts When Models Are Right (LIWC framework)
- **Jiang & Hyland (2025)** — Metadiscourse in ChatGPT essays

---

### GRUP A — Opening Ritual *(zona: opening_zone)*
*Sumber: Observasi discourse per model + Hyland (2005) framing markers*

| ID | Marker | Discourse Function | Tipe | Sumber |
|----|--------|-------------------|------|--------|
| A1 | `meta_framing` | Mengumumkan rencana ("Let's break down") tanpa konten soal | Density | Observasi |
| A2 | `data_restatement` | Menyajikan ulang data soal eksplisit (**Given:** section) | Density | Observasi |
| A3 | `deliberative_entry` | Berpikir keras terlihat di pembuka ("Hmm", "Wait") | **Binary** | Observasi |

---

### GRUP B — Reasoning Organization *(zona: full_text)*
*Sumber: Wei et al. (2022) CoT step structure; Observasi per model*

| ID | Marker | Discourse Function | Tipe | Sumber |
|----|--------|-------------------|------|--------|
| B1 | `explicit_step` | Numbered steps dengan label semantik bold (**Step 1:**) | Density | Wei et al. (2022) |
| B2 | `bare_numbered` | Numbered list tanpa label bold | Density | Wei et al. (2022) |
| B3 | `bullet` | Bullet points (simple dan nested) | Density | Biber (1988) register |
| B4 | `prose_chain` | Logical connectives dalam prose (therefore/thus/hence) | Density | Wei et al. (2022) |
| B5 | `algebraic_expo` | Named-operand formula OLMo-3.1 style | **Binary** | Observasi |

---

### GRUP C — Epistemic Stance *(zona: reasoning_zone untuk C1)*
*Sumber: Hyland (2005) hedges/boosters; arXiv:2602.09832 tentativeness*

| ID | Marker | Discourse Function | Tipe | Sumber |
|----|--------|-------------------|------|--------|
| C1 | `hedging` | Ketidakpastian epistemik (maybe/might/possibly) | Density | Hyland (2005) hedges |
| C2 | `boosters` | Kepercayaan diri tinggi (clearly/definitely) | Density | Hyland (2005) boosters |
| C3 | `assumption_flagging` | Asumsi eksplisit (assuming that/note that) | Density | Hyland (2005) |
| C4 | `interp_disambig` | Menyajikan dua interpretasi sebelum memilih | **Binary** | Observasi |

---

### GRUP D — Inferential Connectives *(zona: full_text)*
*Sumber: Park & Lei (2026) high-entropy forking points; Hyland (2005) transitions*

| ID | Marker | Discourse Function | Tipe | Sumber |
|----|--------|-------------------|------|--------|
| D1 | `formal_logical` | Connective formal-akademik (therefore/thus/hence) | Density | Hyland (2005) transitions |
| D2 | `informal_conseq` | "so [subj] [verb]" — inferensi informal conversational | Density | Park & Lei (2026) |
| D3 | `temporal_seq` | Temporal sequencer (now we/then/finally) | Density | Hyland (2005) frame markers |

---

### GRUP E — Mathematical Formatting *(zona: full_text)*
*Sumber: Biber (1988) Dimension 5 Abstract/Non-abstract; Observasi register*

| ID | Marker | Discourse Function | Tipe | Sumber |
|----|--------|-------------------|------|--------|
| E1 | `latex_display` | LaTeX display block \[…\] atau $$…$$ | Density | Biber (1988) D5 |
| E2 | `latex_inline` | LaTeX inline \(…\) | Density | Biber (1988) D5 |
| E3 | `plain_arith` | Plain arithmetic 3 + 4 = 7 | Density | Biber (1988) D5 |
| E4 | `bold_value` | Bold numeric **$18** | Density | Observasi |
| E5 | `boxed_answer` | \boxed{N} — formal closure ritual | Density | Observasi |

---

### GRUP F — Verification & Closure *(zona: closure_zone untuk F1, F4, F5)*
*Sumber: Meek et al. (2025) verification; Observasi per model*

| ID | Marker | Discourse Function | Tipe | Sumber |
|----|--------|-------------------|------|--------|
| F1 | `ritual_verif` | Verifikasi ritual setelah jawaban (OLMo-3.1 style) | Density | Meek et al. (2025) |
| F2 | `mid_revision` | Genuine error detection di tengah reasoning | **Binary** | Meek et al. (2025) |
| F3 | `interp_verif` | Verifikasi pemahaman soal, bukan aritmetika | **Binary** | Observasi |
| F4 | `nl_closure` | Natural language closure dengan nama karakter | **Binary** | Observasi |
| F5 | `formal_closure` | Bold answer / formal closure | Density | Observasi |

---

### GRUP G — Hyland Interactional/Interactive Markers *(tambahan dari audit)*
*Sumber: Hyland (2005) — diperlukan untuk konsistensi dengan framework yang dikutip*

| ID | Marker | Discourse Function | Tipe | Sumber |
|----|--------|-------------------|------|--------|
| G1 | `attitude_markers` | Stance evaluatif (importantly/surprisingly/unfortunately) | **Binary** | Hyland (2005) attitude |
| G2 | `engagement_markers` | Reader-directing (note that/keep in mind) | **Binary** | Hyland (2005) engagement |
| G3 | `code_glosses` | Reformulasi/elaborasi (in other words/for example/i.e.) | Density | Hyland (2005) code gloss |
| G4 | `inclusive_we` | We reasoning-functional (we need to/we get/let us) | Density | Biber D1 + Hyland self-mentions |
| G5 | `first_person_I` | First-person deliberation (I think/I realize) di reasoning | **Binary** | Biber D1 + Hyland self-mentions |
| G6 | `nominalization` | Proses nominalized (calculation/multiplication) | Density | Biber (1988) D5 |

> **Catatan G1 & G2:** Sangat sparse (<0.5%) karena domain matematika bukan argumentative writing.  
> Jiang & Hyland (2025) menemukan ChatGPT juga jarang menggunakan attitude & engagement markers  
> dibanding human writers — ini sendiri adalah temuan yang relevan untuk TA.

---

### GRUP V — Verbosity Markers *(computed features)*
*Sumber: Meek et al. (2025) — verbosity sebagai formal CoT property*

| ID | Marker | Discourse Function | Tipe | Sumber |
|----|--------|-------------------|------|--------|
| V1 | `word_count` | Total kata per respons | Numeric | Meek et al. (2025) |
| V2 | `sentence_count` | Jumlah kalimat per respons | Numeric | Meek et al. (2025) |
| V3 | `avg_sentence_len` | Rata-rata panjang kalimat (kata/kalimat) | Numeric | Meek et al. (2025) |

> **V1–V3 adalah outcome variables** dalam analisis persona, bukan hanya denominator normalisasi.  
> Meek et al. (2025) secara formal mendefinisikan verbosity sebagai properti CoT tersendiri.

---

### Aggregate Scores

| Score | Formula | Interpretasi |
|-------|---------|-------------|
| `formal_math_register` | E1 + E2 + E5 | Register matematis formal (Biber D5) |
| `informal_math_register` | E3 + E4 | Register matematis informal |
| `epistemic_complexity` | C1 + C2 + C3 | Total stance epistemik |
| `inferential_connectives` | D1 + D2 + D3 | Total connective language |
| `hyland_interactional` | G1 + G2 + G3 + G4 + G5 (binary sum) | Metadiscourse interaksional Hyland |
| `liwc_reasoning_quality` | L_causation + L_differentiation + L_insight | Proxy kualitas reasoning (LIWC) |


In [ ]:
# ── Marker Patterns & Extraction ────────────────────────────────────────────
# Taksonomi: 8 grup, 32 marker (19 density + 13 binary)
# Setiap marker memiliki citation eksplisit ke sumber teori

import re, numpy as np

# ═══════════════════════════════════════════════════════════════
# GRUP A — Opening Ritual  (Hyland 2005 framing + Observasi)
# ═══════════════════════════════════════════════════════════════

_A1 = re.compile(
    r"^(let'?s\s+(break\s+(this|it)|go\s+step|solve\s+this|work\s+through|find)|"
    r"here'?s?\s+how\s+to\s+solve|let\s+me\s+solve|i'?ll\s+solve|"
    r"we\s+can\s+solve|to\s+solve\s+this\s+(problem|step))",
    re.I)
# Hyland (2005) frame markers — "announces plan without content"

_A2 = re.compile(
    r"(\*{1,2}given[:\s\*]|\*{1,2}known[:\s]|given\s+information[:\s]|"
    r"the\s+problem\s+(gives?|states?|says?|tells?\s+us)\b|"
    r"from\s+the\s+problem\b|we\s+are\s+(given|told)\b|"
    r"the\s+given\s+(data|information|values?)\b)",
    re.I | re.MULTILINE)

_A3 = re.compile(
    r"^(okay[,\s]+let'?s\s+see|hmm+[,\s]|wait[,!\s]+(?!the\s+answer)|"
    r"let\s+me\s+(think|re.?read|parse)\b|alright[,\s]+so\b)",
    re.I)  # Binary — thinking models only

# ═══════════════════════════════════════════════════════════════
# GRUP B — Reasoning Organization  (Wei et al. 2022)
# ═══════════════════════════════════════════════════════════════

_B1 = re.compile(r'\*{1,2}Step\s*\d+[:\s\*]|\*{1,2}\d+\.\s+\*{1,2}[A-Z]', re.I)
_B2 = re.compile(r'(?m)^\s*\d+[.)]\s+(?!\*{1,2}[A-Z]|\*{1,2}Step)[a-z]', re.I)
_B3 = re.compile(r'(?m)^\s*[-*•]\s+\w', re.I)
_B4 = re.compile(
    r'\b(therefore\b|thus\b|hence\b|which\s+means\b|this\s+means\b|'
    r'this\s+gives(?:\s+us)?\b|it\s+follows\s+that\b)\b', re.I)
_B5 = re.compile(
    r'[A-Z][a-z\s]{1,20}(?:used|needed|earned|sold|remaining|total|left|made|cost|spent)\s*'
    r'=\s*[A-Z][a-z\s]{1,20}', re.I)  # Binary — OLMo-3.1 algebraic exposition

# ═══════════════════════════════════════════════════════════════
# GRUP C — Epistemic Stance  (Hyland 2005 hedges/boosters)
# ═══════════════════════════════════════════════════════════════

_C1 = re.compile(
    r'\b(maybe\b|perhaps\b|might(?!\s+want|\s+also\s+want)|possibly\b|'
    r'it\s+seems?\b|appears?\s+to\b|i\s+think\s+(?:this|that|the|it)\b|'
    r'not\s+sure\s+(?:if|whether)\b|could\s+(?:be|mean)\b)', re.I)
# Zone: reasoning_zone — menghindari echo prompt

_C2 = re.compile(
    r'\b(clearly\b|obviously\b|definitely\b|certainly\b|'
    r'must\s+be\b|of\s+course\b|undoubtedly\b)', re.I)
# Hyland (2005) boosters

_C3 = re.compile(
    r"\b(assuming\s+(?:that|the|this)\b|i'?ll\s+assume\b|let'?s?\s+assume\b|"
    r"we\s+(?:can\s+)?assume\b|note\s+that\b|per\s+the\s+problem\b|as\s+stated\b)", re.I)
# Hyland (2005) hedges — interpretive transparency

_C4 = re.compile(
    r'(alternatively\b|another\s+(interpretation|way\s+to\s+read)\b|'
    r'this\s+could\s+mean\b|there\s+are\s+two\s+ways\b|'
    r'if\s+(?:this|the\s+problem)\s+means?\b)', re.I)
# Binary — rare (~2%)

# ═══════════════════════════════════════════════════════════════
# GRUP D — Inferential Connectives
# (Hyland 2005 transitions; Park & Lei 2026 forking points)
# ═══════════════════════════════════════════════════════════════

_D1 = re.compile(
    r'\b(therefore\b|thus\b|hence\b|consequently\b|'
    r'it\s+follows\s+that\b|as\s+a\s+result\b)', re.I)
# Hyland (2005) transitions — formal register

_D2 = re.compile(
    r'(?<!\w)so\s+(?:she|he|they|it|we|janet|the|this|that|each|every|'
    r'john|mary|tom|bob|alice|[a-z]{3,10})\s+(?:has|have|makes?|gets?|earns?|'
    r'needs?|can|will|is|are)\b', re.I)
# Park & Lei (2026) — informal consequence chain, prose signature

_D3 = re.compile(
    r"\b(now\s+(?:we|let'?s|i|,)|then\s+(?:we|,)|next[,\s]|"
    r"finally[,\s]|after\s+that\b|lastly\b)", re.I)
# Hyland (2005) frame markers — temporal sequencing

# ═══════════════════════════════════════════════════════════════
# GRUP E — Mathematical Formatting  (Biber 1988 D5)
# ═══════════════════════════════════════════════════════════════

_E1 = re.compile(r'\\\[[\s\S]{1,500}?\\\]|\$\$[\s\S]{1,500}?\$\$', re.I)
# Biber (1988) D5 abstract/non-abstract — formal mathematical register

_E2 = re.compile(r'\\\([^)]{1,150}\\\)|\$(?!\$)[^$\n]{1,80}\$(?!\$)', re.I)
# Biber (1988) D5 — formal embedded

_E3 = re.compile(r'(?<!\*)\b\d[\d,\.]*\s*[+\-×x÷/]\s*\d[\d,\.]*\s*=\s*\d[\d,\.]*(?!\*)', re.I)
# Biber (1988) D5 — informal-functional register

_E4 = re.compile(r'\*\*\$?\d[\d,\.]*(?:\s*(?:eggs?|hours?|days?|km|miles?|dollars?|cents?))?\*\*', re.I)
# Observasi — bold numeric emphasis

_E5 = re.compile(r'\\(?:boxed|bbox)\{[^}]{1,60}\}')
# Observasi — closure ritual formal

# ═══════════════════════════════════════════════════════════════
# GRUP F — Verification & Closure  (Meek et al. 2025)
# ═══════════════════════════════════════════════════════════════

_F1 = re.compile(
    r"\b(let'?s\s+(?:check|verify|double.?check|confirm)\b|"
    r"we\s+can\s+(?:verify|check|confirm)\b|"
    r"(?:makes?\s+sense\b|checks?\s+out\b|this\s+is\s+correct\b))", re.I)
# Meek et al. (2025) — ritual verification closure

_F2 = re.compile(
    r'\b(wait[,!]?\s+(?:that|this|but|no|let|actually)|'
    r'actually[,!]?\s+(?:no|wait|that|i\s+made)|'
    r'let\s+me\s+(?:re.?calculate|re.?check|reconsider)\b|'
    r'i\s+(?:made\s+an?\s+error|was\s+wrong)\b|'
    r"that\s+(?:can\'?t\s+be|doesn\'?t\s+(?:look|seem)\s+right)\b)", re.I)
# Meek et al. (2025) — genuine error detection mid-reasoning; Binary

_F3 = re.compile(
    r'\b(re.?reading\s+the\s+problem\b|let\s+me\s+re.?read\b|'
    r"wait[,]?\s+the\s+problem\s+says?\b|"
    r'hmm[,\s]+(?:the|this|so|wait)\b|'
    r're.?check(?:ing)?\s+the\s+(problem|question)\b)', re.I)
# Observasi — interpretive verification; Binary

_F4 = re.compile(
    r'(?:so\s+)?[A-Z][a-z]+\s+(?:makes?|earns?|gets?|has|will\s+(?:have|get|earn|make|spend))'
    r'\s+\*{0,2}\$?\d[\d,\.]*\*{0,2}', re.I)
# Observasi — NL closure; Binary

_F5 = re.compile(
    r'(\*{0,2}(?:final\s+)?answer\s*[:\-]\*{0,2}\s*\*{0,2}\$?\d|'
    r'\*{1,2}(?:the\s+)?(?:final\s+)?answer\s+is\b|'
    r'#{1,3}\s*✅\s*(?:answer|result)|\\boxed\{)', re.I)
# Observasi — formal closure

# ═══════════════════════════════════════════════════════════════
# GRUP G — Hyland Interactional/Interactive Markers
# (Hyland 2005 — diperlukan untuk konsistensi framework)
# ═══════════════════════════════════════════════════════════════

_G1 = re.compile(
    r'\b(importantly\b|surprisingly\b|unfortunately\b|fortunately\b|'
    r'crucially\b|remarkably\b|interestingly\b|essentially\b|critically\b)\b', re.I)
# Hyland (2005) attitude markers — evaluative stance
# CATATAN: ~0% di math domain — ini sendiri adalah temuan
# Jiang & Hyland (2025): ChatGPT juga jarang gunakan attitude markers vs human
# Binary — model yang menggunakannya adalah outlier informatif

_G2 = re.compile(
    r'\b(note\s+that\b|note\s+here\b|it\s+is\s+(?:worth|important)\s+to\s+note\b|'
    r'keep\s+in\s+mind\b|it\s+should\s+be\s+noted\b|worth\s+noting\b|'
    r'recall\s+that\b|as\s+(?:we\s+can\s+see|noted)\b)\b', re.I)
# Hyland (2005) engagement markers — reader-directing
# Binary — rare tapi reader-awareness yang bermakna

_G3 = re.compile(
    r'\b(in\s+other\s+words\b|that\s+is\s+(?:to\s+say\b|,)?|i\.e\.?\b|'
    r'for\s+example\b|e\.g\.?\b|namely\b|such\s+as\b|'
    r'to\s+put\s+it\s+(?:simply|differently)\b|'
    r'which\s+(?:means|is\s+to\s+say)\b)\b', re.I)
# Hyland (2005) code glosses — reformulation/elaboration

_G4 = re.compile(
    r'\b(we\s+(?:need\s+to\b|can\s+(?:now\b|see\b|find\b|calculate\b|determine\b)|'
    r'get\b|have\b|know\b|find\b|calculate\b|must\b|should\b|want\s+to\b)|'
    r'let\s+us\b|our\s+(?:answer\b|result\b|calculation\b|total\b|goal\b|final\b))\b', re.I)
# Biber D1 + Hyland self-mentions — inclusive-we reasoning function

_G5 = re.compile(
    r'\b(i\s+(?:think\b|believe\b|realize\b|notice\b|find\s+that\b|'
    r'calculated\b|get\b|was\s+wrong\b|made\s+an\s+error\b))\b', re.I)
# Biber D1 + Hyland self-mentions — first-person deliberation
# Zone: reasoning_zone; Binary

_G6 = re.compile(
    r'\b(calculation\b|multiplication\b|addition\b|subtraction\b|division\b|'
    r'determination\b|evaluation\b|estimation\b|approximation\b|'
    r'simplification\b|substitution\b|computation\b|derivation\b|'
    r'measurement\b|comparison\b)\b', re.I)
# Biber (1988) D5 — nominalization: abstract information packaging

# ═══════════════════════════════════════════════════════════════
# LIWC-ALIGNED MARKERS  (arXiv:2602.09832)
# "Faulty reasoning 5x more likely to rely on epistemic hedging"
# ═══════════════════════════════════════════════════════════════

_L_CAUS = re.compile(
    r'\b(because\b|since\s+(?:she|he|they|it|we|the|each|every|there)\b|'
    r'causes?\b|leads?\s+to\b|results?\s+in\b|due\s+to\b|'
    r'as\s+a\s+(?:result|consequence)\b|therefore\b|thus\b|hence\b)\b', re.I)
# LIWC Causation — reason-giving language
# Overlap dengan D1 disengaja: D1 mengukur formal connective,
# L_causation mengukur broader causal language termasuk "because/since"

_L_DIFF = re.compile(
    r'\b(but\b|however\b|although\b|despite\b|whereas\b|'
    r'nevertheless\b|on\s+the\s+other\s+hand\b|in\s+contrast\b|'
    r'instead\b|rather\s+than\b|even\s+though\b)\b', re.I)
# LIWC Differentiation — contrastive language; high in wrong answers

_L_INS = re.compile(
    r'\b(i\s+(?:think\b|know\b|realize\b|notice\b|understand\b|see\b)|'
    r'(?:this|it)\s+(?:shows?\b|reveals?\b|indicates?\b|suggests?\b|implies?\b)|'
    r'(?:we\s+can\s+see\b|it\s+is\s+clear\b|clearly\s+shows?\b))\b', re.I)
# LIWC Insight — metacognitive language; Zone: reasoning_zone; Binary

# ═══════════════════════════════════════════════════════════════
# HELPERS
# ═══════════════════════════════════════════════════════════════

def _cnt(text, pat):
    return len(pat.findall(str(text))) if isinstance(text, str) else 0

def _has(text, pat):
    return int(bool(pat.search(str(text)))) if isinstance(text, str) else 0

# ═══════════════════════════════════════════════════════════════
# APPLY ALL MARKERS
# ═══════════════════════════════════════════════════════════════

print("Extracting all 32 markers from response_content...")

# Grup A
df["A1_meta_framing_s"]       = df["opening_zone"].apply(lambda x: _cnt(x, _A1))
df["A2_data_restatement_s"]   = df["text"].apply(lambda x: _cnt(x, _A2))
df["A3_deliberative_entry"]   = df["opening_zone"].apply(lambda x: _has(x, _A3))

# Grup B
df["B1_explicit_step_s"]      = df["text"].apply(lambda x: _cnt(x, _B1))
df["B2_bare_numbered_s"]      = df["text"].apply(lambda x: _cnt(x, _B2))
df["B3_bullet_s"]             = df["text"].apply(lambda x: _cnt(x, _B3))
df["B4_prose_chain_s"]        = df["text"].apply(lambda x: _cnt(x, _B4))
df["B5_algebraic_expo"]       = df["reasoning_zone"].apply(lambda x: _has(x, _B5))

# Grup C
df["C1_hedging_s"]            = df["reasoning_zone"].apply(lambda x: _cnt(x, _C1))
df["C2_boosters_s"]           = df["text"].apply(lambda x: _cnt(x, _C2))
df["C3_assumption_s"]         = df["text"].apply(lambda x: _cnt(x, _C3))
df["C4_interp_disambig"]      = df["text"].apply(lambda x: _has(x, _C4))

# Grup D
df["D1_formal_logical_s"]     = df["text"].apply(lambda x: _cnt(x, _D1))
df["D2_informal_conseq_s"]    = df["text"].apply(lambda x: _cnt(x, _D2))
df["D3_temporal_seq_s"]       = df["text"].apply(lambda x: _cnt(x, _D3))

# Grup E
df["E1_latex_display_s"]      = df["text"].apply(lambda x: _cnt(x, _E1))
df["E2_latex_inline_s"]       = df["text"].apply(lambda x: _cnt(x, _E2))
df["E3_plain_arith_s"]        = df["text"].apply(lambda x: _cnt(x, _E3))
df["E4_bold_value_s"]         = df["text"].apply(lambda x: _cnt(x, _E4))
df["E5_boxed_answer_s"]       = df["text"].apply(lambda x: _cnt(x, _E5))

# Grup F
df["F1_ritual_verif_s"]       = df["closure_zone"].apply(lambda x: _cnt(x, _F1))
df["F2_mid_revision"]         = df["reasoning_zone"].apply(lambda x: _has(x, _F2))
df["F3_interp_verif"]         = df["text"].apply(lambda x: _has(x, _F3))
df["F4_nl_closure"]           = df["closure_zone"].apply(lambda x: _has(x, _F4))
df["F5_formal_closure_s"]     = df["closure_zone"].apply(lambda x: _cnt(x, _F5))

# Grup G — Hyland Interactional/Interactive
df["G1_attitude"]             = df["text"].apply(lambda x: _has(x, _G1))
df["G2_engagement"]           = df["text"].apply(lambda x: _has(x, _G2))
df["G3_code_glosses_s"]       = df["text"].apply(lambda x: _cnt(x, _G3))
df["G4_inclusive_we_s"]       = df["text"].apply(lambda x: _cnt(x, _G4))
df["G5_first_person_I"]       = df["reasoning_zone"].apply(lambda x: _has(x, _G5))
df["G6_nominalization_s"]     = df["text"].apply(lambda x: _cnt(x, _G6))

# LIWC-aligned
df["L_causation_s"]           = df["text"].apply(lambda x: _cnt(x, _L_CAUS))
df["L_differentiation_s"]     = df["text"].apply(lambda x: _cnt(x, _L_DIFF))
df["L_insight"]               = df["reasoning_zone"].apply(lambda x: _has(x, _L_INS))

# Grup V — Verbosity (Meek et al. 2025)
df["V_word_count"]       = df["word_count"]
df["V_sentence_count"]   = df["text"].apply(
    lambda t: len([s for s in re.split(r'(?<=[.!?])\s+', str(t)) if s.strip()]))
df["V_avg_sentence_len"] = np.where(
    df["V_sentence_count"] > 0,
    df["V_word_count"] / df["V_sentence_count"], np.nan)

# ── Density normalization (per 100 kata) ─────────────────────────────────────
DENSITY_MARKERS = [
    "A1_meta_framing","A2_data_restatement",
    "B1_explicit_step","B2_bare_numbered","B3_bullet","B4_prose_chain",
    "C1_hedging","C2_boosters","C3_assumption",
    "D1_formal_logical","D2_informal_conseq","D3_temporal_seq",
    "E1_latex_display","E2_latex_inline","E3_plain_arith","E4_bold_value","E5_boxed_answer",
    "F1_ritual_verif","F5_formal_closure",
    "G3_code_glosses","G4_inclusive_we","G6_nominalization",
    "L_causation","L_differentiation",
]
BINARY_MARKERS = [
    "A3_deliberative_entry","B5_algebraic_expo","C4_interp_disambig",
    "F2_mid_revision","F3_interp_verif","F4_nl_closure",
    "G1_attitude","G2_engagement","G5_first_person_I","L_insight",
]

for m in DENSITY_MARKERS:
    df[f"{m}_d"] = np.where(
        df["word_count"] > 0,
        df[f"{m}_s"] / df["word_count"] * 100, np.nan)

# ── Aggregate scores ──────────────────────────────────────────────────────────
df["formal_math_register"]     = df["E1_latex_display_d"] + df["E2_latex_inline_d"] + df["E5_boxed_answer_d"]
df["informal_math_register"]   = df["E3_plain_arith_d"] + df["E4_bold_value_d"]
df["epistemic_complexity"]     = df["C1_hedging_d"] + df["C2_boosters_d"] + df["C3_assumption_d"]
df["inferential_connectives"]  = df["D1_formal_logical_d"] + df["D2_informal_conseq_d"] + df["D3_temporal_seq_d"]
df["hyland_interactional"]     = df["G1_attitude"] + df["G2_engagement"] + (df["G3_code_glosses_d"]>0).astype(int) + (df["G4_inclusive_we_d"]>0).astype(int)
df["liwc_reasoning_quality"]   = df["L_causation_d"] + df["L_differentiation_d"] + df["L_insight"]
df["verbosity_score"]          = df["V_sentence_count"]  # primary verbosity signal

ALL_DENSITY_COLS = [f"{m}_d" for m in DENSITY_MARKERS]
ALL_BINARY_COLS  = BINARY_MARKERS

print(f"✅ Markers extracted: {len(DENSITY_MARKERS)} density + {len(BINARY_MARKERS)} binary = {len(DENSITY_MARKERS)+len(BINARY_MARKERS)} total")
print(f"\nGrup G (Hyland) coverage:")
for g in ["G1_attitude","G2_engagement","G3_code_glosses_d","G4_inclusive_we_d","G5_first_person_I","G6_nominalization_d"]:
    col = g if g in df.columns else f"{g}"
    if col in df.columns:
        pct = (df[col] > 0).mean() * 100
        print(f"  {col:25s}: {pct:.1f}% docs")
print(f"\nLIWC coverage:")
for l in ["L_causation_d","L_differentiation_d","L_insight"]:
    col = l
    if col in df.columns:
        pct = (df[col] > 0).mean() * 100
        print(f"  {col:25s}: {pct:.1f}% docs")
print(f"\nVerbosity (mean per model):")
print(df.groupby("model_label")["V_sentence_count"].mean().round(1).sort_values(ascending=False))


## 3. Validasi Marker — Coverage, Sparsity, Variance

In [ ]:
# ── Marker Validation ────────────────────────────────────────────────────────

rows = []
for m in DENSITY_MARKERS:
    col = f"{m}_d"
    vals = df[col].dropna()
    pct_present = (df[f"{m}_s"] > 0).mean() * 100
    rows.append({
        "Marker": m, "Type": "Density",
        "%Docs_present": round(pct_present, 1),
        "Mean_density": round(vals.mean(), 4),
        "Std_density": round(vals.std(), 4),
        "CV": round(vals.std()/vals.mean() if vals.mean()>0 else 0, 2),
        "Status": "⚠ Sparse (<2%)" if pct_present < 2 else
                  ("⚠ Low (<5%)" if pct_present < 5 else
                  ("✅ OK" if vals.std() > 0.001 else "⚠ Near-constant")),
    })

for m in BINARY_MARKERS:
    vals = df[m]
    pct = vals.mean() * 100
    rows.append({
        "Marker": m, "Type": "Binary",
        "%Docs_present": round(pct, 1),
        "Mean_density": round(vals.mean(), 4),
        "Std_density": round(vals.std(), 4),
        "CV": "-",
        "Status": "⚠ Sparse (<2%)" if pct < 2 else
                  ("⚠ Low (<5%)" if pct < 5 else "✅ OK"),
    })

val_df = pd.DataFrame(rows)
print(val_df.to_string(index=False))

# Visualisasi: prevalence bar chart
fig, ax = plt.subplots(figsize=(12, 7))
colors = ["#e74c3c" if "Sparse" in r["Status"] else "#f39c12" if "Low" in r["Status"] or "constant" in r["Status"]
          else "#27ae60" for _, r in val_df.iterrows()]
bars = ax.barh(val_df["Marker"], val_df["%Docs_present"], color=colors, edgecolor="white", linewidth=0.5)
ax.axvline(2, color="red", linestyle="--", linewidth=1, label="2% threshold (sparse)")
ax.axvline(5, color="orange", linestyle="--", linewidth=1, label="5% threshold (low)")
ax.set_xlabel("% Responses Containing Marker", fontsize=11)
ax.set_title("Marker Prevalence Across Full Dataset\n(Red = Sparse <2%, Orange = Low <5%, Green = OK)", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
for bar, val in zip(bars, val_df["%Docs_present"]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f"{val:.1f}%", va="center", fontsize=8)
plt.tight_layout()
plt.show()

print("\n⚠ Sparse markers digunakan sebagai binary flag — interpretasikan sebagai probabilitas kemunculan, bukan density.")

# ── Catatan untuk Grup G (Hyland) dan LIWC ───────────────────────────────────
# G1 attitude dan G2 engagement sangat sparse (<0.5%)
# Ini BUKAN masalah implementasi — ini adalah temuan:
# Jiang & Hyland (2025) menemukan ChatGPT menggunakan lebih sedikit
# attitude & engagement markers dibanding human writers.
# Dalam math domain, kelangkaan ini bahkan lebih ekstrem.
# Interpretasi: "absence of attitude markers is itself a discourse finding"

# Check per model untuk G1/G2 — model mana yang menggunakannya?
print("\nGrup G per model (binary %):")
for g in ["G1_attitude","G2_engagement","G3_code_glosses_d","G4_inclusive_we_d","G5_first_person_I"]:
    col = g if "_d" not in g else g
    if col in df.columns:
        per_model = df.groupby("model_label")[col].mean().mul(100).round(2)
        print(f"\n  {col}:")
        print(per_model.sort_values(ascending=False).to_string())

print("\n\nLIWC Causation + Differentiation per model (density):")
for l in ["L_causation_d","L_differentiation_d"]:
    if l in df.columns:
        pm = df.groupby("model_label")[l].mean().round(4)
        print(f"\n  {l}:")
        print(pm.sort_values(ascending=False).to_string())


## 4. Qualitative Example Table — Representative Discourse Samples

In [ ]:
# ── Qualitative Examples per Marker ─────────────────────────────────────────
# Menampilkan contoh `response_content` yang mewakili tiap marker

import textwrap

EXAMPLE_MARKERS = {
    "A1_meta_framing_d":     ("Meta Framing Opener", "A1"),
    "A2_data_restatement_d": ("Data Restatement", "A2"),
    "B1_explicit_step_d":    ("Explicit Step Enum.", "B1"),
    "B3_bullet_d":           ("Bullet Hierarchy", "B3"),
    "B4_prose_chain_d":      ("Prose Chain", "B4"),
    "C1_hedging_d":          ("Epistemic Hedging", "C1"),
    "D1_formal_logical_d":   ("Formal Logical Conn.", "D1"),
    "D2_informal_conseq_d":  ("Informal Consequence", "D2"),
    "E1_latex_display_d":    ("LaTeX Display", "E1"),
    "E3_plain_arith_d":      ("Plain Arithmetic", "E3"),
    "E5_boxed_answer_d":     ("Boxed Answer", "E5"),
    "F4_nl_closure":         ("NL Closure", "F4"),
}

print(f"{'ID':>3} {'Marker':22s} {'Model':18s} {'Persona':18s} {'Example (first 150 chars)':50s}")
print("="*115)

for col, (label, mid) in EXAMPLE_MARKERS.items():
    if col in df.columns:
        subset = df[df[col] > 0.1] if col.endswith("_d") else df[df[col] == 1]
        if len(subset) == 0:
            continue
        row = subset.sample(1, random_state=42).iloc[0]
        snippet = row["text"][:160].replace("\n", " ").replace("\r", "")
        snippet = textwrap.shorten(snippet, width=80)
        model_s = str(row.get("model_label",""))[:17]
        persona_s = str(row["persona"])[:17]
        print(f"{mid:>3} {label:22s} {model_s:18s} {persona_s:18s} {snippet}")
    print()


## 5. Profil Linguistic Marker Per Model

In [ ]:
# ── Per-Model Aggregation ────────────────────────────────────────────────────

def agg_model(group):
    r = {"n": len(group), "wc_mean": group["word_count"].mean()}
    for col in ALL_DENSITY_COLS:
        r[col] = group[col].mean()
    for col in ALL_BINARY_COLS:
        r[col] = group[col].mean() * 100  # sebagai %
    r["formal_math_register"]    = group["formal_math_register"].mean()
    r["informal_math_register"]  = group["informal_math_register"].mean()
    r["epistemic_complexity"]    = group["epistemic_complexity"].mean()
    r["inferential_connectives"] = group["inferential_connectives"].mean()
    return pd.Series(r)

model_profile = df.groupby("model_label").apply(agg_model).reset_index()
model_profile = model_profile.sort_values("B3_bullet_d", ascending=False)

# ── Heatmap: semua density markers per model ──────────────────────────────────
heat_cols = ALL_DENSITY_COLS + ["A3_deliberative_entry","F2_mid_revision","F4_nl_closure"]
heat_data = model_profile.set_index("model_label")[heat_cols].astype(float)

# Label yang lebih pendek
heat_data.columns = [c.replace("_d","").replace("_s","") for c in heat_data.columns]

# Normalize per kolom untuk komparabilitas
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
heat_norm = pd.DataFrame(
    scaler.fit_transform(heat_data),
    index=heat_data.index,
    columns=heat_data.columns
)

fig, ax = plt.subplots(figsize=(18, 8))
sns.heatmap(heat_norm, ax=ax, cmap="YlOrRd", linewidths=0.3, linecolor="#ddd",
            annot=heat_data.round(2), fmt=".2f", annot_kws={"size":7},
            cbar_kws={"label":"Normalized (0=min, 1=max per marker)"})
ax.set_title("Linguistic Marker Profile per Model\n(Color normalized per marker; annotation = raw density/% for binary)",
             fontsize=12, fontweight="bold", pad=15)
ax.set_xlabel("Marker", fontsize=10)
ax.set_ylabel("Model", fontsize=10)
ax.tick_params(axis="x", rotation=45, labelsize=8)
ax.tick_params(axis="y", labelsize=9)
plt.tight_layout()
plt.show()

# ── Bar chart: panjang respons per model ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
wc_data = model_profile.sort_values("wc_mean", ascending=False)
bars = ax.bar(wc_data["model_label"], wc_data["wc_mean"],
              color=["#3498db" if "Think" in m else "#95a5a6" for m in wc_data["model_label"]])
ax.set_ylabel("Rata-rata Word Count")
ax.set_title("Rata-rata Panjang Respons per Model\n(Biru = Thinking model)", fontsize=11, fontweight="bold")
ax.tick_params(axis="x", rotation=45, labelsize=9)
for bar, val in zip(bars, wc_data["wc_mean"]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
            f"{val:.0f}", ha="center", fontsize=8)
plt.tight_layout()
plt.show()


## 6. Analisis Berdasarkan Arsitektur dan Tipe Model

In [ ]:
# ── Dense vs MoE ─────────────────────────────────────────────────────────────

def agg_group(df_sub, key_cols=None):
    if key_cols is None:
        key_cols = ALL_DENSITY_COLS + ALL_BINARY_COLS +                    ["formal_math_register","informal_math_register","epistemic_complexity","inferential_connectives","wc_mean"]
    r = {"n": len(df_sub)}
    for col in ALL_DENSITY_COLS:
        r[col] = df_sub[col].mean()
    for col in ALL_BINARY_COLS:
        r[col] = df_sub[col].mean() * 100
    r["formal_math"]   = df_sub["formal_math_register"].mean()
    r["informal_math"] = df_sub["informal_math_register"].mean()
    r["epistemic"]     = df_sub["epistemic_complexity"].mean()
    r["inferential"]   = df_sub["inferential_connectives"].mean()
    r["wc"]            = df_sub["word_count"].mean()
    return r

# Arch comparison
arch_data = {k: agg_group(g) for k,g in df.groupby("arch")}
arch_df = pd.DataFrame(arch_data).T

# Type comparison
type_data = {k: agg_group(g) for k,g in df.groupby("model_type")}
type_df = pd.DataFrame(type_data).T

# ── Plot: Arch — key markers ──────────────────────────────────────────────────
KEY_MARKERS = ["E1_latex_display_d","E3_plain_arith_d","C1_hedging_d",
               "D2_informal_conseq_d","B3_bullet_d","F2_mid_revision"]
KEY_LABELS  = ["LaTeX Display","Plain Arith","Hedging (C1)",
               "Informal Conseq (D2)","Bullet (B3)","Mid-Revision (F2, %)"]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Arch
arch_plot = arch_df[KEY_MARKERS].astype(float)
arch_plot.columns = KEY_LABELS
arch_plot.T.plot(kind="bar", ax=axes[0], colormap="tab10", edgecolor="white")
axes[0].set_title("Dense vs. MoE — Key Markers", fontsize=11, fontweight="bold")
axes[0].set_xlabel(""); axes[0].set_ylabel("Density / %")
axes[0].tick_params(axis="x", rotation=35, labelsize=9)
axes[0].legend(fontsize=9)

# Type
type_plot = type_df[KEY_MARKERS].astype(float)
type_plot.columns = KEY_LABELS
type_plot.T.plot(kind="bar", ax=axes[1], colormap="Set2", edgecolor="white")
axes[1].set_title("Thinking vs Instruct vs Others — Key Markers", fontsize=11, fontweight="bold")
axes[1].set_xlabel(""); axes[1].set_ylabel("")
axes[1].tick_params(axis="x", rotation=35, labelsize=9)
axes[1].legend(fontsize=9)

plt.suptitle("Linguistic Marker Comparison by Architecture and Model Type", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# ── Math Register: formal vs informal ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(arch_df))
w = 0.35
ax.bar(["Dense (formal)","MoE (formal)"], [arch_df.loc["Dense","formal_math"], arch_df.loc["MoE","formal_math"]],
       color="#2980b9", label="Formal Math Register (E1+E2+E5)")
ax2 = ax.twinx()
ax2.bar(["Dense (informal)","MoE (informal)"],
        [arch_df.loc["Dense","informal_math"], arch_df.loc["MoE","informal_math"]],
        color="#e74c3c", alpha=0.7, label="Informal Math Register (E3+E4)")
ax.set_title("Formal vs Informal Mathematical Register: Dense vs MoE", fontsize=11, fontweight="bold")
ax.set_ylabel("Formal Math Density", color="#2980b9")
ax2.set_ylabel("Informal Math Density", color="#e74c3c")
plt.tight_layout()
plt.show()


## 7. Analisis Persona Gender — Man / Woman / Neutral & Implicit vs Explicit

In [ ]:
# ── Persona Group Mapping ────────────────────────────────────────────────────

# 5 kelompok utama untuk analisis gender
PERSONA_5 = {
    "neutral":       df["persona"] == "neutral",
    "man_explicit":  df["persona"] == "man_explicit_1",
    "man_implicit":  df["persona"].isin(["man_implicit_1","man_implicit_2","man_implicit_3"]),
    "woman_explicit":df["persona"] == "woman_explicit_1",
    "woman_implicit":df["persona"].isin(["woman_implicit_1","woman_implicit_2","woman_implicit_3"]),
}

persona5_agg = {}
for pg, mask in PERSONA_5.items():
    persona5_agg[pg] = agg_group(df[mask])

p5_df = pd.DataFrame(persona5_agg).T
PERSONA_ORDER = ["neutral","man_explicit","man_implicit","woman_explicit","woman_implicit"]
PERSONA_COLORS = {"neutral":"#95a5a6","man_explicit":"#2980b9","man_implicit":"#7fb3d3",
                  "woman_explicit":"#e74c3c","woman_implicit":"#f1948a"}

# ── Plot 1: Key markers across 5 persona groups ───────────────────────────────
PLOT_MARKERS = ["C1_hedging_d","D2_informal_conseq_d","E1_latex_display_d",
                "B3_bullet_d","F5_formal_closure_d","A1_meta_framing_d"]
PLOT_LABELS  = ["Hedging (C1)","Informal Conseq (D2)","LaTeX Display (E1)",
                "Bullet (B3)","Formal Closure (F5)","Meta Framing (A1)"]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, (col, label) in enumerate(zip(PLOT_MARKERS, PLOT_LABELS)):
    vals = [p5_df.loc[pg, col] for pg in PERSONA_ORDER]
    colors = [PERSONA_COLORS[pg] for pg in PERSONA_ORDER]
    bars = axes[i].bar(PERSONA_ORDER, vals, color=colors, edgecolor="white", linewidth=0.8)
    axes[i].set_title(f"{label}", fontsize=10, fontweight="bold")
    axes[i].set_xlabel("")
    axes[i].set_ylabel("Density per 100 kata")
    axes[i].tick_params(axis="x", rotation=35, labelsize=8)
    for bar, val in zip(bars, vals):
        axes[i].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                     f"{val:.3f}", ha="center", fontsize=7)

plt.suptitle("Linguistic Marker Distribution: 5 Persona Groups (neutral/man_exp/man_imp/woman_exp/woman_imp)",
             fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# ── Plot 2: Heatmap semua marker × 5 persona ─────────────────────────────────
heat_p5 = p5_df[ALL_DENSITY_COLS].astype(float)
heat_p5.columns = [c.replace("_d","") for c in heat_p5.columns]
heat_p5 = heat_p5.reindex(PERSONA_ORDER)

# Normalize per kolom
heat_p5_norm = pd.DataFrame(
    scaler.fit_transform(heat_p5.T).T,
    index=heat_p5.index, columns=heat_p5.columns
)

fig, ax = plt.subplots(figsize=(18, 5))
sns.heatmap(heat_p5_norm, ax=ax, cmap="RdBu_r", center=0.5,
            annot=heat_p5.round(3), fmt=".3f", annot_kws={"size":7},
            linewidths=0.3, linecolor="#ddd",
            cbar_kws={"label":"Normalized density"})
ax.set_title("Marker Density per Persona Group (5 groups)\nAnnotation = raw density; Color = normalized",
             fontsize=11, fontweight="bold", pad=12)
ax.tick_params(axis="x", rotation=45, labelsize=8)
ax.tick_params(axis="y", labelsize=9)
plt.tight_layout()
plt.show()


## 8. Implicit vs. Explicit Persona Comparison

In [ ]:
# ── Implicit vs Explicit — semua marker ─────────────────────────────────────

impl_mask = df["persona"].isin(["man_implicit_1","man_implicit_2","man_implicit_3",
                                 "woman_implicit_1","woman_implicit_2","woman_implicit_3"])
expl_mask = df["persona"].isin(["man_explicit_1","woman_explicit_1"])
ntrl_mask = df["persona"] == "neutral"

ie_data = {
    "Implicit": agg_group(df[impl_mask]),
    "Explicit": agg_group(df[expl_mask]),
    "Neutral":  agg_group(df[ntrl_mask]),
}
ie_df = pd.DataFrame(ie_data).T

# Delta: implicit vs explicit
delta_ie = ie_df.loc["Implicit"] - ie_df.loc["Explicit"]
delta_ntrl = ie_df.loc["Implicit"] - ie_df.loc["Neutral"]

# ── Plot: Delta bar chart ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax_i, (delta, title) in enumerate([
    (delta_ie[ALL_DENSITY_COLS], "Δ Implicit − Explicit (density)"),
    (delta_ntrl[ALL_DENSITY_COLS], "Δ Implicit − Neutral (density)"),
]):
    delta_sorted = delta.sort_values(ascending=True)
    labels = [c.replace("_d","") for c in delta_sorted.index]
    colors = ["#e74c3c" if v < 0 else "#27ae60" for v in delta_sorted.values]
    axes[ax_i].barh(labels, delta_sorted.values, color=colors, edgecolor="white")
    axes[ax_i].axvline(0, color="black", linewidth=0.8)
    axes[ax_i].set_title(title, fontsize=11, fontweight="bold")
    axes[ax_i].set_xlabel("Delta density (per 100 kata)")
    axes[ax_i].tick_params(axis="y", labelsize=8)

plt.suptitle("Implicit Persona: Pergeseran Marker dibanding Explicit dan Neutral",
             fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# ── Per-model: implicit vs explicit scatter ──────────────────────────────────
MODEL_LABELS = df["model_label"].dropna().unique()
KEY_DELTA_MARKERS = ["C1_hedging_d","D2_informal_conseq_d","E1_latex_display_d","B3_bullet_d"]

fig, axes = plt.subplots(1, len(KEY_DELTA_MARKERS), figsize=(18, 6))
for ax_i, m in enumerate(KEY_DELTA_MARKERS):
    x_vals, y_vals, labels = [], [], []
    for model in MODEL_LABELS:
        impl = df[(df["model_label"]==model) & impl_mask][m].mean()
        expl = df[(df["model_label"]==model) & expl_mask][m].mean()
        x_vals.append(expl); y_vals.append(impl); labels.append(model)
    axes[ax_i].scatter(x_vals, y_vals, s=60, alpha=0.8, color="#3498db")
    mn = min(min(x_vals), min(y_vals))
    mx = max(max(x_vals), max(y_vals))
    axes[ax_i].plot([mn,mx],[mn,mx], "k--", linewidth=0.8, alpha=0.5)
    for lbl, x, y in zip(labels, x_vals, y_vals):
        axes[ax_i].annotate(lbl[:10], (x,y), fontsize=6, ha="left")
    axes[ax_i].set_xlabel("Explicit density"); axes[ax_i].set_ylabel("Implicit density")
    axes[ax_i].set_title(m.replace("_d",""), fontsize=9, fontweight="bold")

plt.suptitle("Implicit vs Explicit per Model — Points above diagonal = implicit lebih tinggi",
             fontsize=11, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


## 9. Delta Analysis — Pergeseran Marker dari Kondisi Neutral

In [ ]:
# ── Delta per (model × persona_group) ───────────────────────────────────────

neutral_per_model = (
    df[df["persona"]=="neutral"]
    .groupby("model_label")[ALL_DENSITY_COLS]
    .mean()
)

persona_groups_for_delta = {
    "man_explicit":  df["persona"] == "man_explicit_1",
    "man_implicit":  df["persona"].isin(["man_implicit_1","man_implicit_2","man_implicit_3"]),
    "woman_explicit":df["persona"] == "woman_explicit_1",
    "woman_implicit":df["persona"].isin(["woman_implicit_1","woman_implicit_2","woman_implicit_3"]),
}

# Hitung delta per model per persona group
delta_records = []
for pg, mask in persona_groups_for_delta.items():
    sub = df[mask].groupby("model_label")[ALL_DENSITY_COLS].mean()
    for model in sub.index:
        if model not in neutral_per_model.index:
            continue
        for col in ALL_DENSITY_COLS:
            delta = sub.loc[model, col] - neutral_per_model.loc[model, col]
            delta_records.append({
                "model": model, "persona_group": pg,
                "marker": col.replace("_d",""), "delta": delta
            })

delta_df = pd.DataFrame(delta_records)

# ── Heatmap: delta per marker × persona group (averaged across models) ────────
delta_pivot = delta_df.groupby(["persona_group","marker"])["delta"].mean().unstack("marker")
delta_pivot = delta_pivot.reindex(["man_explicit","man_implicit","woman_explicit","woman_implicit"])

fig, ax = plt.subplots(figsize=(18, 5))
vmax = delta_pivot.abs().max().max()
sns.heatmap(delta_pivot, ax=ax, cmap="RdBu_r", center=0, vmin=-vmax, vmax=vmax,
            annot=True, fmt=".3f", annot_kws={"size":7},
            linewidths=0.3, linecolor="#ddd",
            cbar_kws={"label":"Δ density (persona − neutral)"})
ax.set_title("Delta Marker Density: Persona − Neutral Baseline\n"
             "Merah = marker meningkat dari neutral; Biru = menurun",
             fontsize=11, fontweight="bold", pad=12)
ax.tick_params(axis="x", rotation=45, labelsize=8)
plt.tight_layout()
plt.show()

# ── Per-model delta heatmap untuk marker terpilih ────────────────────────────
KEY_DELTA = ["C1_hedging","D2_informal_conseq","E1_latex_display","B3_bullet",
             "A1_meta_framing","F5_formal_closure"]

for pg in ["woman_implicit","man_implicit"]:
    sub_delta = delta_df[delta_df["persona_group"]==pg]
    sub_pivot = sub_delta[sub_delta["marker"].isin(KEY_DELTA)].pivot(
        index="model", columns="marker", values="delta")
    vmax = sub_pivot.abs().max().max()
    fig, ax = plt.subplots(figsize=(12, 7))
    sns.heatmap(sub_pivot, ax=ax, cmap="RdBu_r", center=0, vmin=-vmax, vmax=vmax,
                annot=True, fmt=".3f", annot_kws={"size":8},
                linewidths=0.3, linecolor="#ddd",
                cbar_kws={"label":"Δ density"})
    ax.set_title(f"Delta Marker: {pg} − Neutral (per model, 6 key markers)",
                 fontsize=11, fontweight="bold", pad=10)
    ax.tick_params(axis="x", rotation=35, labelsize=9)
    ax.tick_params(axis="y", labelsize=8)
    plt.tight_layout()
    plt.show()


## 10. Analisis Berdasarkan Kategori Ukuran Model

In [ ]:
# ── Size Category Comparison ─────────────────────────────────────────────────

SIZE_ORDER = ["Small (7–10B)","Medium (15–50B)","Large (70–100B)",
              "Very Large (100–200B)","Ultra Large (200B+)","Proprietary"]

size_agg = {}
for scat, grp in df.groupby("size_cat"):
    size_agg[scat] = agg_group(grp)

size_df = pd.DataFrame(size_agg).T
size_df = size_df.reindex([s for s in SIZE_ORDER if s in size_df.index])

# ── Plot: LaTeX vs Plain Arith trend by size ─────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

markers_to_plot = [
    (["E1_latex_display_d","E3_plain_arith_d","E5_boxed_answer_d"],
     ["LaTeX Display","Plain Arith","Boxed Answer"], "Mathematical Register by Size"),
    (["C1_hedging_d","C2_boosters_d","C3_assumption_d"],
     ["Hedging","Boosters","Assumption Flag"], "Epistemic Stance by Size"),
    (["D1_formal_logical_d","D2_informal_conseq_d","B4_prose_chain_d"],
     ["Formal Logical","Informal Conseq","Prose Chain"], "Inferential Style by Size"),
]

for ax_i, (cols, lbls, title) in enumerate(markers_to_plot):
    plot_data = size_df[cols].astype(float)
    plot_data.columns = lbls
    plot_data.plot(kind="bar", ax=axes[ax_i], colormap="viridis", edgecolor="white")
    axes[ax_i].set_title(title, fontsize=10, fontweight="bold")
    axes[ax_i].set_xlabel("")
    axes[ax_i].tick_params(axis="x", rotation=35, labelsize=8)
    axes[ax_i].legend(fontsize=8)

plt.suptitle("Linguistic Marker Trends by Model Size Category",
             fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


## 11. Binary Marker Analysis — Probabilitas Kemunculan per Kondisi

In [ ]:
# ── Binary Markers: probabilitas kemunculan ──────────────────────────────────

binary_persona = {}
for pg, mask in {**PERSONA_5, "implicit_all": impl_mask, "explicit_all": expl_mask}.items():
    sub = df[mask]
    binary_persona[pg] = {m: sub[m].mean()*100 for m in ALL_BINARY_COLS}

bin_df = pd.DataFrame(binary_persona).T

# Plot: grouped bar per binary marker
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, bm in enumerate(ALL_BINARY_COLS):
    pg_order = ["neutral","man_explicit","man_implicit","woman_explicit","woman_implicit"]
    vals = [bin_df.loc[pg, bm] for pg in pg_order if pg in bin_df.index]
    pgs  = [pg for pg in pg_order if pg in bin_df.index]
    colors = [PERSONA_COLORS.get(pg, "#7f8c8d") for pg in pgs]
    axes[i].bar(pgs, vals, color=colors, edgecolor="white")
    axes[i].set_title(bm, fontsize=9, fontweight="bold")
    axes[i].set_ylabel("% responses")
    axes[i].tick_params(axis="x", rotation=35, labelsize=8)
    for j, (bar_v, val) in enumerate(zip(pgs, vals)):
        axes[i].text(j, val+0.1, f"{val:.1f}%", ha="center", fontsize=7)

plt.suptitle("Binary Marker Probability per Persona Group\n(% of responses containing marker)",
             fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# Per-model binary marker rates
bin_model = df.groupby("model_label")[ALL_BINARY_COLS].mean().mul(100).round(2)
print("\nBinary Marker Rate (%) per Model:")
print(bin_model.sort_values("F2_mid_revision", ascending=False).to_string())


## 12. Panduan Interpretasi untuk Penulisan TA

### Temuan Utama yang Bisa Diklaim

**1. Register Matematis (Grup E)**  
`E1_latex_display` adalah pembeda terkuat antar model. Model dengan LaTeX tinggi (DeepSeek V3.2, GPT-oss 120B)
menunjukkan sensitivitas persona yang lebih rendah. Gunakan `formal_math_register` vs `informal_math_register`
sebagai sumbu "formality axis" dalam analisis.

**2. Inferensi Implisit vs Formal (D1 vs D2)**  
`D2_informal_conseq` ("so she has...") adalah proxy terbaik untuk "prose vs format reasoning".
Distribusinya tidak overlap dengan arsitektur — LLaMA tinggi D2 terlepas dari ukurannya.

**3. Epistemic Stance (C1) hanya bermakna pada Thinking Models**  
`C1_hedging` di reasoning_zone hanya signifikan untuk Qwen3-Think dan OLMo-Think.
Pada instruct models, kemunculannya terlalu sparse untuk klaim.

**4. Binary markers sebagai model signature**  
`F2_mid_revision` hampir eksklusif di thinking models. `F4_nl_closure` eksklusif di Gemma/LLaMA.
Gunakan sebagai pembeda model, bukan sebagai pembeda persona.

**5. Delta dari neutral > absolute value**  
Untuk klaim efek persona: selalu gunakan `Δ = persona − neutral`.
Absolute value marker berbeda-beda per model sehingga tidak bisa dibandingkan antar-model.

### Batasan yang Harus Dinyatakan
- B5 (algebraic_expo) sangat sparse (~0.3%), hanya OLMo-3.1
- F1 (ritual_verif) sangat sparse (<0.5%), hanya OLMo-3.1 dan kadang Qwen3-Think
- C1 (hedging) di reasoning_zone mungkin masih ada echo dari prompt implisit
- E1 vs E2 saling overlap untuk model yang menggunakan kedua dialek LaTeX
